# DiffAIM Colab Runner

[Liqin-Wang/DiffAIM](https://github.com/Liqin-Wang/DiffAIM) — Stable Diffusion 1.5 + ensemble FR surrogate (IR152/IRSE50/MobileFace/FaceNet) + structure-preserving regularization. Transfer-based black-box attack。

**vs 我們的 DiffProtect：**
- 擴散模型大 5×（SD 1.5 vs DiffAE）
- Surrogate 從 1 個 (IRSE50) → 4 個 ensemble
- 多了 structural + LPIPS perceptual loss

**流程：**
1. Clone repo + 安裝套件
2. 下載 SD 1.5（從 HuggingFace，~4GB）
3. 下載 FR 權重（從 DiffProtect 同一個 zip）
4. （選用）上傳自己的 source/target，或用 repo 自帶範例
5. 跑 `main.py`，產出對抗圖
6. 預覽 + 下載

> ⚠️ 這份不用 numpy 1.26（DiffAIM 不依賴 InsightFace 0.7.3）。要驗證 InsightFace transfer 請另開 notebook 或 restart runtime 切換環境。

## 0. Colab GPU 檢查

In [ ]:
import torch, sys
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info(0)
    print(f'VRAM free / total: {free/1e9:.1f}GB / {total/1e9:.1f}GB')
else:
    raise RuntimeError('請切換 Colab runtime → GPU')

## 1. Clone DiffAIM

In [ ]:
%cd /content
!rm -rf DiffAIM
!git clone https://github.com/Liqin-Wang/DiffAIM.git
%cd DiffAIM
!git rev-parse --short HEAD
!ls

## 2. 安裝套件

DiffAIM 的 requirements 釘 `torch==2.1.1+cu118` 跟 `numpy==2.3.2`。Colab 已經有更新版 torch + cuda，所以**不要動 torch**，只裝 diffusers / lpips。

In [ ]:
!python -m pip install -q diffusers==0.31.0 lpips==0.1.4
!python -m pip install -q transformers accelerate safetensors
print('Done')

## 3. 下載 Stable Diffusion v1.5

DiffAIM 預設讀 `./pretrained/stable-diffusion-v1-5`。從 HuggingFace 下載（~4GB，看網速 2-5 分鐘）。

In [ ]:
from pathlib import Path
import os

SD_DIR = Path('/content/DiffAIM/pretrained/stable-diffusion-v1-5')
SD_DIR.parent.mkdir(parents=True, exist_ok=True)

if not (SD_DIR / 'model_index.json').exists():
    print('Downloading SD 1.5 from HuggingFace (~4GB)...')
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id='runwayml/stable-diffusion-v1-5',
        local_dir=str(SD_DIR),
        local_dir_use_symlinks=False,
        allow_patterns=['*.json', '*.txt', 'unet/*', 'vae/*', 'text_encoder/*',
                        'tokenizer/*', 'scheduler/*', 'safety_checker/*', 'feature_extractor/*'],
        ignore_patterns=['*.bin', '*.ckpt'],  # 只抓 safetensors
    )
    print('SD 1.5 downloaded')
else:
    print('SD 1.5 already exists')
!ls {SD_DIR}

## 4. 下載 FR 權重

需要 `pretrained/{ir152, irse50, facenet, mobile_face}.pth`。我們從 DiffProtect 的同一個 Google Drive zip 抓出來（同來源，AMT-GAN 那批）。

In [ ]:
from pathlib import Path
import shutil, glob, subprocess

PRETRAINED = Path('/content/DiffAIM/pretrained')
PRETRAINED.mkdir(parents=True, exist_ok=True)
required = ['ir152.pth', 'irse50.pth', 'facenet.pth', 'mobile_face.pth']

missing = [n for n in required if not (PRETRAINED / n).exists()]
if missing:
    print(f'Missing: {missing}, downloading...')
    subprocess.run('rm -rf /content/diffprotect_assets_extract && mkdir -p /content/diffprotect_assets_extract', shell=True)
    if not Path('/content/diffprotect_assets.zip').exists():
        subprocess.run("gdown --fuzzy 'https://drive.google.com/file/d/19_Y0jR789BGciogjjoGtWNEv-5QBiCB7/view?usp=sharing' -O /content/diffprotect_assets.zip", shell=True)
    subprocess.run('unzip -q -o /content/diffprotect_assets.zip -d /content/diffprotect_assets_extract', shell=True)
    for fname in required:
        matches = glob.glob(f'/content/diffprotect_assets_extract/**/{fname}', recursive=True)
        if matches:
            shutil.copy2(matches[0], PRETRAINED / fname)
            print(f'  copied {fname}')

print('\nFR weight status:')
for n in required:
    p = PRETRAINED / n
    print(f'  {n}: {"OK" if p.exists() else "MISSING"} ({p.stat().st_size / 1e6:.1f}MB)' if p.exists() else f'  {n}: MISSING')

## 5. 上傳自己的 source/target（可選）

DiffAIM repo 自帶 `dataset/src/` 5 張 + `dataset/target/` 1 張範例。
- 不上傳：直接跑 repo 自帶範例
- 上傳：把你的圖蓋過去

如果要用自己的，跑下面這格上傳；不用就直接跳到 cell 6。

In [ ]:
USE_CUSTOM = False  # 改 True 上傳自己的 source/target

if USE_CUSTOM:
    from google.colab import files
    import shutil
    from pathlib import Path

    SRC_DIR = Path('/content/DiffAIM/dataset/src')
    TGT_DIR = Path('/content/DiffAIM/dataset/target')

    # 清空
    for d in [SRC_DIR, TGT_DIR]:
        if d.exists():
            for f in d.iterdir():
                f.unlink()

    print('請上傳 source 自拍照（可多張）')
    up = files.upload()
    for n, _ in up.items():
        if Path(n).suffix.lower() in ['.jpg', '.jpeg', '.png']:
            shutil.move(n, SRC_DIR / n)

    print('請上傳 target face')
    up = files.upload()
    for n, _ in up.items():
        if Path(n).suffix.lower() in ['.jpg', '.jpeg', '.png']:
            shutil.move(n, TGT_DIR / n)

print('\nsrc folder:')
!ls /content/DiffAIM/dataset/src
print('target folder:')
!ls /content/DiffAIM/dataset/target

## 6. 跑攻擊（main.py）

預設 `--target_model IR152` — 換成 IRSE50 / FaceNet / MobileFace 也行。
SD 1.5 比 DiffAE 重很多，**每張 source 大概 1-3 分鐘**。num=5 大約 5-15 分鐘。

In [ ]:
import subprocess, time
from pathlib import Path

TARGET_MODEL = 'IR152'   # IR152 / IRSE50 / FaceNet / MobileFace
NUM = 5                  # 處理幾張 source
SAVE_DIR = 'results'

cmd = (
    f'cd /content/DiffAIM && python main.py '
    f'--device_id 0 '
    f'--image_size 256 '
    f'--num {NUM} '
    f'--target_model {TARGET_MODEL} '
    f'--save {SAVE_DIR}'
)
print(f'$ {cmd}\n')
t0 = time.time()
proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
elapsed = time.time() - t0

print(proc.stdout[-3000:])
if proc.returncode != 0:
    print('\nSTDERR:')
    print(proc.stderr[-3000:])
    raise RuntimeError(f'main.py exit={proc.returncode}')
print(f'\nDone in {elapsed:.1f}s')
!ls /content/DiffAIM/results/{TARGET_MODEL}/

## 7. 預覽結果

In [ ]:
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt

results_dir = Path(f'/content/DiffAIM/results/{TARGET_MODEL}')
src_dir = Path('/content/DiffAIM/dataset/src')
adv_imgs = sorted(results_dir.glob('*.png'))
print(f'Generated {len(adv_imgs)} adversarial images')

# 並列顯示 source vs adv
n = min(len(adv_imgs), 5)
fig, axes = plt.subplots(2, n, figsize=(4*n, 8))
if n == 1: axes = axes.reshape(2, 1)
for i, adv_p in enumerate(adv_imgs[:n]):
    src_name = adv_p.stem  # adv 檔名應對 src 檔名
    src_candidates = list(src_dir.glob(f'{src_name}.*'))
    if src_candidates:
        axes[0, i].imshow(Image.open(src_candidates[0]))
        axes[0, i].set_title(f'src: {src_name}', fontsize=9)
    axes[1, i].imshow(Image.open(adv_p))
    axes[1, i].set_title(f'adv: {adv_p.name}', fontsize=9)
    axes[0, i].axis('off'); axes[1, i].axis('off')
plt.suptitle(f'DiffAIM attack via {TARGET_MODEL} ensemble')
plt.tight_layout()
plt.show()

## 8. 下載結果

In [ ]:
from google.colab import files
zip_path = '/content/diffaim_results.zip'
!rm -f {zip_path}
!cd /content/DiffAIM && zip -qr {zip_path} results dataset/src dataset/target
print(f'Packed: {zip_path}')
files.download(zip_path)

## 常見問題

- **OOM**：SD 1.5 + 4 個 FR ensemble 吃 VRAM，T4 (15GB) 通常 OK 但可能很滿。降 `--num 1` 試試。
- **torch / diffusers 不相容**：repo 用舊版 torch 2.1.1+cu118，Colab 用更新版；如果遇到 API 改名，把 diffusers 降版到 0.27 試試。
- **想要更強攻擊**：調 `--steps` 上限、`--eps` 上限、降 `--lpips_loss_weight`（放掉視覺正則，攻擊更兇）。
- **要對 InsightFace 驗證 transfer**：本 notebook 沒裝 InsightFace（numpy 2 環境）。換個 notebook（feature_space_alignment.ipynb 那種環境）載入 adv 圖再驗證。